# EfficientNetB0 — Train a cat-versus-dog classifier

Use case: adapt an existing CNN to two categories in your own image workflow. Transfer learning reuses learned features. We freeze EfficientNetB0 and train a small classifier on top of its extracted features.

Run the cells from top to bottom. Each notebook is independent.


## Step 1 — Install packages
Use Python 3.11 or 3.12 as a straightforward training environment, or Google Colab. Install into the active notebook kernel. Restart the kernel after the first installation if required. Internet is needed initially for packages, sample images/data and ImageNet weights. A GPU is optional; CPU runs can be slower. Model downloads can be hundreds of MB.


In [ ]:
%pip install tensorflow numpy matplotlib scikit-learn pillow


## Step 2 — Import libraries
TensorFlow runs the neural network. Keras provides the model API. NumPy handles arrays and Matplotlib displays pictures.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__)


## Step 3 — Load labeled pictures
We select cats (label 3) and dogs (label 5). CIFAR-10 is a low-resolution teaching dataset, not a deployment benchmark.


In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.ravel()
test_labels = test_labels.ravel()
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]
print("Training images:", train_images.shape)
print("Test images:", test_images.shape)
plt.figure(figsize=(9, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i])
    plt.title(class_names[train_labels[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


## Step 4 — Create separate training, validation and test sets
Training changes weights, validation guides model selection, and the test set is used once for the final evaluation. Here we use small balanced samples for a laptop demo.


In [ ]:
from sklearn.model_selection import train_test_split
selected_train = []
selected_test = []
for label in [3, 5]:
    selected_train.extend(np.where(train_labels == label)[0][:300])
    selected_test.extend(np.where(test_labels == label)[0][:100])
images = train_images[selected_train]
labels = (train_labels[selected_train] == 5).astype("float32")
x_train, x_val, y_train, y_val = train_test_split(
    images, labels, test_size=0.2, stratify=labels, random_state=42)
x_test = test_images[selected_test]
y_test = (test_labels[selected_test] == 5).astype("float32")
print("Train/validation/test:", len(x_train), len(x_val), len(x_test))


## Step 5 — Extract frozen EfficientNet features
EfficientNetB0 includes its input rescaling. Pass float pixel values in the 0–255 range; do not divide by 255 first. Features are computed once to avoid repeatedly running the CNN during classifier training.


In [ ]:
extractor = keras.applications.EfficientNetB0(
    weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))
extractor.trainable = False

def extract_features(images):
    parts = []
    for start in range(0, len(images), 16):
        batch = tf.image.resize(images[start:start + 16], (224, 224))
        parts.append(extractor.predict(batch, verbose=0))
    return np.concatenate(parts)

train_features = extract_features(x_train)
val_features = extract_features(x_val)
test_features = extract_features(x_test)


## Step 6 — Train the new classifier
A sigmoid output gives the model score for dog. Binary cross-entropy measures error. Early stopping restores weights from the best validation loss. Only the classifier is trained.


In [ ]:
classifier = keras.Sequential([
    keras.layers.Input(shape=(train_features.shape[1],)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation="sigmoid")
])
classifier.compile(optimizer=keras.optimizers.Adam(0.001),
                   loss="binary_crossentropy", metrics=["accuracy"])
history = classifier.fit(train_features, y_train,
                         validation_data=(val_features, y_val),
                         epochs=12, batch_size=32,
                         callbacks=[keras.callbacks.EarlyStopping(
                             patience=3, restore_best_weights=True)])
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch index")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.show()


## Step 7 — Evaluate and display a prediction
The test score is measured on unseen test images. Results vary; no accuracy is guaranteed.


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
scores = classifier.predict(test_features, verbose=0).ravel()
predicted = (scores >= 0.5).astype(int)
print(classification_report(y_test.astype(int), predicted, target_names=["cat", "dog"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test.astype(int), predicted, display_labels=["cat", "dog"])
plt.show()
image_index = 42
plt.imshow(x_test[image_index])
plt.title(f"Actual: {'dog' if y_test[image_index] else 'cat'} | Predicted: {'dog' if predicted[image_index] else 'cat'}")
plt.axis("off")
plt.show()


## Step 8 — Save and predict one local picture
Save the complete pipeline so the CNN and classifier stay together. Supply an RGB image resized to 224 × 224 with values 0–255.


In [ ]:
inputs = keras.Input(shape=(224, 224, 3))
outputs = classifier(extractor(inputs, training=False))
full_model = keras.Model(inputs, outputs)
full_model.save("cat_dog_efficientnet.keras")

# Optional: replace None with a local filename, for example "my_dog.jpg".
custom_image_path = None
if custom_image_path is not None:
    picture = keras.utils.load_img(custom_image_path, target_size=(224, 224))
    batch = np.expand_dims(keras.utils.img_to_array(picture), axis=0)
    dog_score = float(full_model.predict(batch, verbose=0)[0, 0])
    plt.imshow(picture)
    plt.title(f"{'dog' if dog_score >= 0.5 else 'cat'} | dog score: {dog_score:.3f}")
    plt.axis("off")
    plt.show()


## Understand the result
Feature extraction is one form of transfer learning. Fine-tuning would additionally update selected pretrained CNN weights. This binary model always chooses cat or dog, even for unrelated pictures; it has no “other” category.


## Practice
1. Change the input image or selected image index.
2. Explain the output in your own words.
3. Compare the result with the original image and record one limitation.


## References
- [Keras Applications](https://keras.io/api/applications/)
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [Keras Grad-CAM example](https://keras.io/examples/vision/grad_cam/)
